In [ ]:
import numpy as np
import sys
# from gym.envs.toy_text import discrete

import gymnasium as gym
from gymnasium import spaces


# 定义动作空间
up = 0
right = 1
down = 2
left = 3

# 定义宝藏所在状态
done_location = 8


# 定义网格世界环境模型
class GridworldEnv(gym.Env):
    '''定义5x5网格
    [[o,o,o,o,o],
     [o,o,o,T,o],
     [o,o,o,o,o],
     [o,x,o,o,o],
     o,o,o,o,o]]
    x为智能体位置，T宝藏位置'''
    
    def __init__(self, shape=[5, 5]):
        if not isinstance(shape, (list, tuple)) or not len(shape) == 2:
            raise ValueError('shape argument must be a list/tuple of length 2')
        self.shape = shape
        
        nS = np.prod(shape)  # 状态个数，5x5=25
        nA = 4  # 动作个数
        max_Y = shape[0]  # 5
        max_X = shape[1]  # 5
        
        P = {}
        grid = np.arange(nS).reshape(shape)
        it = np.nditer(grid, flags=['multi_index'])
        
        while not it.finished:
            s = it.iterindex
            y, x = it.multi_index
            P[s] = {a: [] for a in range(nA)}
            is_done = lambda s: s == done_location
            reward = 0.0 if is_done(s) else -1.0
            
            if is_done(s):
                P[s][up] = [(1, s, reward, True)]
                P[s][right] = [(1, s, reward, True)]
                P[s][down] = [(1, s, reward, True)]
                P[s][left] = [(1, s, reward, True)]
            else:
                ns_up = s if y == 0 else s - max_X
                ns_right = s if x == (max_X - 1) else s + 1
                ns_down = s if y == (max_Y - 1) else s + max_X
                ns_left = s if x == 0 else s - 1
                P[s][up] = [(1, ns_up, reward, is_done(ns_up))]
                P[s][right] = [(1, ns_right, reward, is_done(ns_right))]
                P[s][down] = [(1, ns_down, reward, is_done(ns_down))]
                P[s][left] = [(1, ns_left, reward, is_done(ns_left))]
            it.iternext()
            
        isd = np.ones(nS) / nS
        self.P = P
        # super(GridworldEnv, self).__init__(nS, nA, P, isd)

        self.nS = nS
        self.nA = nA
        self.isd = isd

        self.observation_space = spaces.Discrete(nS)
        self.action_space = spaces.Discrete(nA)

        self.s = np.random.choice(nS, p=isd)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.s = np.random.choice(self.nS, p=self.isd)
        return self.s, {}
    
    def step(self, action):
        transitions = self.P[self.s][action]
        prob, next_state, reward, done = transitions[0]

        self.s = next_state

        terminated = done
        truncated = False

        return next_state, reward, terminated, truncated, {}


In [2]:
import numpy as np
grid = np.arange(9).reshape((3,3))
it = np.nditer(grid, flags = ['multi_index'])

while not it.finished:
    s = it.iterindex
    y, x= it.multi_index
    print('s,y,x:',s,y,x)
    it.iternext()

s,y,x: 0 0 0
s,y,x: 1 0 1
s,y,x: 2 0 2
s,y,x: 3 1 0
s,y,x: 4 1 1
s,y,x: 5 1 2
s,y,x: 6 2 0
s,y,x: 7 2 1
s,y,x: 8 2 2


In [3]:
shape=[5,5]
nS = np.prod(shape) # 状态个数，5x5=25
print("nS:",nS)

nA = 4 # 动作个数
max_Y = shape[0] # 5
max_X = shape[1] # 5


nS: 25


In [4]:
P={}
grid = np.arange(nS).reshape(shape)
it = np.nditer(grid, flags = ['multi_index'])

print("grid:",grid)
print(' ')

print('it:',it, type(it))

grid: [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]]
 
it: <numpy.nditer object at 0x00000191BD9F6BF0> <class 'numpy.nditer'>


In [5]:
s = it.iterindex
print('s:',s)

y, x= it.multi_index
print("y,x:",y,x)

P[s] = {a:[] for a in range(nA)}
print("P[s]:",P[s])

is_done = lambda s: s == done_location
reward = 0.0 if is_done(s) else -1.0

P[s][up]=[(1,s,reward,True)]

print('P[s][up]:',P[s][up])
print(P[s])

s: 0
y,x: 0 0
P[s]: {0: [], 1: [], 2: [], 3: []}
P[s][up]: [(1, 0, -1.0, True)]
{0: [(1, 0, -1.0, True)], 1: [], 2: [], 3: []}


In [6]:
m=it.iternext()
print(m,type(m))

isd = np.ones(nS)/nS
print(isd)

True <class 'bool'>
[0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04
 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04 0.04]


In [7]:
from gymnasium.utils import seeding
class env:
    def seed(self, seed=None):
        print(seeding.np_random(seed))
        self.np_random, seed = seeding.np_random(seed)
        return [seed]
    
a=env()

print(a.seed(seed=1234))
print(a.seed())

(Generator(PCG64) at 0x18257E77D80, 1234)
[1234]
(Generator(PCG64) at 0x182594A4660, 270480839479221825409659642288392382366)
[201745131240748629667476208987251243386]


In [1]:
hand=[1,2,3,4,5,6,2,3]
p_val=3
mm=sum(1 for i in hand if i == p_val)
print(mm)

2
